[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [15]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [16]:
data_frame = pd.read_csv("spotify_data_clean.csv")

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [17]:
data_frame.isnull().sum()

track_id                 0
track_name               0
track_number             0
track_popularity         0
explicit                 0
artist_name              3
artist_popularity        0
artist_followers         0
artist_genres         3361
album_id                 0
album_name               0
album_release_date       0
album_total_tracks       0
album_type               0
track_duration_min       0
dtype: int64

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [21]:
# Remove Null values
data_frame = data_frame.dropna(subset=['artist_name', 'artist_genres'])
data_frame.isnull().sum()

track_id              0
track_name            0
track_number          0
track_popularity      0
explicit              0
artist_name           0
artist_popularity     0
artist_followers      0
artist_genres         0
album_id              0
album_name            0
album_release_date    0
album_total_tracks    0
album_type            0
track_duration_min    0
dtype: int64

In [ ]:
# Replace Null values with the mean value for the column
data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
data_frame.isnull().sum()

KeyError: 'Target'

#### Remove Duplicates

Duplicate data can have detrimental effects on your machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [24]:
data_frame.duplicated().sum()

np.int64(0)

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [25]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

np.int64(0)

#### Replace data

We can run a lambda function on a column to modify its values. For a simple example, let’s convert the Sex to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [26]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda x: x.lower())
data_frame['SEX'].head()

KeyError: 'SEX'

We can check that there are no data entry errors by the `unique()` method call.

In [ ]:
data_frame['SEX'].unique()

<StringArray>
['female', 'male', 'girl']
Length: 3, dtype: str

In [ ]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: 'male' if gender.lower() == 'male' else 'female')
data_frame['SEX'].unique()

<StringArray>
['female', 'male']
Length: 2, dtype: str

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [39]:
# get the inter-quartile range on the blood pressure column
print(data_frame['track_popularity'].describe())
Q3 = data_frame["track_popularity"].quantile(0.998)
IQR = Q3 - Q1
print(f'Outliers are a track popularity above {Q3}')

count    5221.000000
mean       53.555258
std        22.987208
min         0.000000
25%        41.000000
50%        59.000000
75%        71.000000
max        99.000000
Name: track_popularity, dtype: float64
Outliers are a track popularity above 93.0


In [40]:
# Filter blood pressure within the acceptable range
data_frame = data_frame[
    (data_frame["track_popularity"] <= Q3)
]
print(data_frame["track_popularity"].describe())

count    5215.000000
mean       53.506807
std        22.955894
min         0.000000
25%        41.000000
50%        59.000000
75%        71.000000
max        93.000000
Name: track_popularity, dtype: float64


#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [43]:
print(data_frame["track_popularity"].describe())
print(data_frame["artist_popularity"].describe())
print(data_frame["artist_followers"].describe())

count    5215.000000
mean       53.506807
std        22.955894
min         0.000000
25%        41.000000
50%        59.000000
75%        71.000000
max        93.000000
Name: track_popularity, dtype: float64
count    5215.000000
mean       74.566251
std        17.174687
min         2.000000
25%        63.000000
50%        79.000000
75%        88.000000
max       100.000000
Name: artist_popularity, dtype: float64
count    5.215000e+03
mean     3.376855e+07
std      4.465486e+07
min      4.000000e+00
25%      1.298544e+06
50%      1.096042e+07
75%      4.771699e+07
max      1.455421e+08
Name: artist_followers, dtype: float64


In [ ]:
scale_feature_TP = "track_popularity"
scale_feature_AP = "artist_popularity"
scale_feature_AF = "artist_followers"

# the minimum value with space for outliers
MIN_TP = 0
MIN_AP = 2
MIN_AF = 4

# the maximum value with space for outliers
MAX_TP = 93
MAX_AP = 100
MAX_AF = 145542136

# scale features
data_frame[scale_feature_TP] = [(X - MIN_TP) / (MAX_TP - MIN_TP) for X in data_frame[scale_feature_TP]]
data_frame[scale_feature_AP] = [(X - MIN_AP) / (MAX_AP - MIN_AP) for X in data_frame[scale_feature_AP]]
data_frame[scale_feature_AF] = [(X - MIN_AF) / (MAX_AF - MIN_AF) for X in data_frame[scale_feature_AF]]

data_frame.describe()

,track_number,track_popularity,artist_popularity,artist_followers,album_total_tracks,track_duration_min
count,5215.000000,5215.000000,5215.000000,5215.000000,5215.000000,5215.000000
mean,6.469223,0.575342,0.740472,0.232019,15.899712,3.565126
std,6.327266,0.246838,0.175252,0.306817,13.107231,1.133671
min,1.000000,0.000000,0.000000,0.000000,1.000000,0.140000
25%,1.000000,0.440860,0.622449,0.008922,10.000000,2.900000
50%,5.000000,0.634409,0.785714,0.075308,14.000000,3.510000
75%,10.000000,0.763441,0.877551,0.327857,19.000000,4.080000
max,102.000000,1.000000,1.000000,1.000000,181.000000,13.510000


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [47]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)